# 01 — Data Ingestion

**Threat Incident Intelligence Platform**

This notebook:
1. Loads the ThreatCluster dataset **only** via the Hugging Face `datasets` library (no local files).
2. Validates and normalizes fields.
3. Cleans text and entities using shared `src` utilities.
4. Inserts documents into MongoDB in efficient batches.
5. Creates useful indexes.


In [1]:
## 1. Imports & Configuration

import sys
from pathlib import Path

# Ensure project root is on PYTHONPATH so `src` can be imported
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hp\Documents\GitHub\cyberthreat_intelligence_ai


In [2]:
from datetime import datetime
from typing import Any, Dict, List, Optional

from datasets import load_dataset
from tqdm.auto import tqdm

from src.config import settings, validate_config, COLLECTIONS
from src.db import (
    ping_database,
    get_incidents_collection,
    get_db,
    close_connection,
    insert_many_incidents,
    count_incidents,
)
from src.preprocessing import (
    clean_text,
    normalize_entities_document,
)
from src.utils import (
    setup_logging,
    parse_date,
    safe_get,
    safe_float,
    safe_int,
    ensure_list,
    batch_iterator,
    timer,
    is_valid_cluster_id,
)

logger = setup_logging("INFO", name="ingestion")
validate_config()
print("Config OK")

[config] MongoDB URI loaded (db=threat_intelligence)
[config] Collections: ['incidents', 'incident_features', 'predictions', 'models_metadata']
Config OK


In [3]:
## 2. MongoDB Connectivity Check

if not ping_database():
    raise RuntimeError(
        "Cannot reach MongoDB. Check MONGODB_URI in .env and ensure the server is running."
    )
print("MongoDB is reachable")
print(f"Database : {settings.mongodb_db}")
print(f"Collection: {COLLECTIONS['incidents']}")

MongoDB is reachable
Database : threat_intelligence
Collection: incidents


In [4]:
## 3. Load Dataset via Hugging Face `datasets` 

DATASET_ID = "threatcluster/threat-incident-clusters"

with timer("Loading Hugging Face dataset"):
    # load_dataset pulls data through the HF API / cache managed by the library
    ds = load_dataset(DATASET_ID, split="train")

print(f"Dataset loaded: {len(ds):,} records")
print(f"Features     : {list(ds.features.keys())}")
print("\nSample record keys:", list(ds[0].keys()) if len(ds) else "(empty)")

README.md: 0.00B [00:00, ?B/s]

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\datasets--threatcluster--threat-incident-clusters. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed

data.jsonl:   0%|          | 0.00/25.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19205 [00:00<?, ? examples/s]

[timer] Loading Hugging Face dataset: 19.24s
Dataset loaded: 19,205 records
Features     : ['cluster_id', 'title', 'summary', 'article_count', 'threat_score', 'severity_score', 'credibility_score', 'urgency_level', 'keywords', 'first_reported', 'last_reported', 'entities', 'sources', 'url', 'source_count']

Sample record keys: ['cluster_id', 'title', 'summary', 'article_count', 'threat_score', 'severity_score', 'credibility_score', 'urgency_level', 'keywords', 'first_reported', 'last_reported', 'entities', 'sources', 'url', 'source_count']


In [6]:
## 4. Field Validation & Normalization Helpers

REQUIRED_FIELDS = ["cluster_id", "title", "summary"]

def normalize_record(raw: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Validate and normalize a single raw dataset record into a clean
    MongoDB document. Returns None if the record is unusable.
    """
    cluster_id = safe_get(raw, "cluster_id")
    if not cluster_id or not isinstance(cluster_id, str):
        return None
    cluster_id = cluster_id.strip()
    if not cluster_id:
        return None

    title = safe_get(raw, "title", "")
    summary = safe_get(raw, "summary", "")

    # At least one of title / summary should have content
    if not title and not summary:
        return None

    # Preserve original text + store a cleaned version for NLP
    title_clean = clean_text(title, remove_punctuation=False)
    summary_clean = clean_text(summary, remove_punctuation=False)

    # Entities
    entities_raw = safe_get(raw, "entities", {})
    entities = normalize_entities_document(entities_raw)

    # Keywords → always a list of strings
    keywords = ensure_list(safe_get(raw, "keywords", []))
    keywords = [str(k).strip() for k in keywords if k and str(k).strip()]

    # Sources
    sources = ensure_list(safe_get(raw, "sources", []))
    sources = [str(s).strip() for s in sources if s and str(s).strip()]

    # Scores
    threat_score = safe_float(safe_get(raw, "threat_score"), default=None)
    severity_score = safe_float(safe_get(raw, "severity_score"), default=None)
    credibility_score = safe_float(safe_get(raw, "credibility_score"), default=None)

    # Counts
    source_count = safe_int(safe_get(raw, "source_count"), default=len(sources))
    article_count = safe_int(safe_get(raw, "article_count"), default=0)

    # Urgency
    urgency_level = safe_get(raw, "urgency_level")
    if urgency_level is not None:
        urgency_level = str(urgency_level).strip().lower() or None

    # Dates
    first_reported = parse_date(safe_get(raw, "first_reported"))
    last_reported = parse_date(safe_get(raw, "last_reported"))

    # URL
    url = safe_get(raw, "url")
    if url:
        url = str(url).strip()

    doc = {
        "cluster_id": cluster_id,
        "title": title,
        "title_clean": title_clean,
        "summary": summary,
        "summary_clean": summary_clean,
        "entities": entities,
        "keywords": keywords,
        "sources": sources,
        "source_count": source_count,
        "article_count": article_count,
        "threat_score": threat_score,
        "severity_score": severity_score,
        "credibility_score": credibility_score,
        "urgency_level": urgency_level,
        "first_reported": first_reported,
        "last_reported": last_reported,
        "url": url,
        "ingested_at": datetime.utcnow(),
    }
    return doc

In [7]:
## 5. Transform All Records

documents: List[Dict[str, Any]] = []
skipped = 0
seen_ids = set()

with timer("Normalizing records"):
    for raw in tqdm(ds, desc="Normalizing"):
        doc = normalize_record(raw)
        if doc is None:
            skipped += 1
            continue

        cid = doc["cluster_id"]
        if cid in seen_ids:
            skipped += 1
            continue
        seen_ids.add(cid)
        documents.append(doc)

print(f"Ready to insert : {len(documents):,}")
print(f"Skipped         : {skipped:,}")
print(f"Unique IDs      : {len(seen_ids):,}")

Normalizing:   0%|          | 0/19205 [00:00<?, ?it/s]

C:\Users\hp\AppData\Local\Temp\ipykernel_8076\2366015258.py:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingested_at": datetime.utcnow(),


[timer] Normalizing records: 13.27s
Ready to insert : 19,204
Skipped         : 1
Unique IDs      : 19,204


In [8]:
## 7. Bulk Insert into MongoDB

BATCH_SIZE = 500
total_inserted = 0
errors = 0

coll = get_incidents_collection()

with timer("Bulk insertion"):
    for batch in tqdm(
        batch_iterator(documents, batch_size=BATCH_SIZE),
        total=(len(documents) + BATCH_SIZE - 1) // BATCH_SIZE,
        desc="Inserting batches",
    ):
        try:
            result = insert_many_incidents(batch, ordered=False)
            total_inserted += len(result.inserted_ids)
        except Exception as exc:
            # ordered=False means some docs may still have been inserted
            logger.warning("Batch insert error (continuing): %s", exc)
            errors += 1

print(f"Inserted : {total_inserted:,}")
print(f"Batch errors: {errors}")
print(f"Collection count now: {count_incidents():,}")

Inserting batches:   0%|          | 0/39 [00:00<?, ?it/s]

[timer] Bulk insertion: 1.54s
Inserted : 19,204
Batch errors: 0
Collection count now: 19,204


In [9]:
## 8. Create Indexes

from pymongo import ASCENDING, DESCENDING

coll = get_incidents_collection()

index_specs = [
    ([("cluster_id", ASCENDING)], {"unique": True, "name": "idx_cluster_id"}),
    ([("urgency_level", ASCENDING)], {"name": "idx_urgency"}),
    ([("threat_score", DESCENDING)], {"name": "idx_threat_score"}),
    ([("severity_score", DESCENDING)], {"name": "idx_severity_score"}),
    ([("first_reported", DESCENDING)], {"name": "idx_first_reported"}),
    ([("last_reported", DESCENDING)], {"name": "idx_last_reported"}),
    ([("source_count", DESCENDING)], {"name": "idx_source_count"}),
    ([("article_count", DESCENDING)], {"name": "idx_article_count"}),
    # Text index for basic keyword search on title + summary
    ([("title", "text"), ("summary", "text"), ("keywords", "text")], {"name": "idx_text"}),
]

existing = {idx["name"] for idx in coll.list_indexes()}

for keys, opts in index_specs:
    name = opts.get("name")
    if name in existing:
        print(f"Index already exists: {name}")
        continue
    try:
        coll.create_index(keys, **opts)
        print(f"Created index: {name}")
    except Exception as exc:
        print(f"Failed to create {name}: {exc}")

print("\nCurrent indexes:")
for idx in coll.list_indexes():
    print(f"  - {idx['name']}: {idx.get('key')}")

Created index: idx_cluster_id
Created index: idx_urgency
Created index: idx_threat_score
Created index: idx_severity_score
Created index: idx_first_reported
Created index: idx_last_reported
Created index: idx_source_count
Created index: idx_article_count
Created index: idx_text

Current indexes:
  - _id_: SON([('_id', 1)])
  - idx_cluster_id: SON([('cluster_id', 1)])
  - idx_urgency: SON([('urgency_level', 1)])
  - idx_threat_score: SON([('threat_score', -1)])
  - idx_severity_score: SON([('severity_score', -1)])
  - idx_first_reported: SON([('first_reported', -1)])
  - idx_last_reported: SON([('last_reported', -1)])
  - idx_source_count: SON([('source_count', -1)])
  - idx_article_count: SON([('article_count', -1)])
  - idx_text: SON([('_fts', 'text'), ('_ftsx', 1)])


In [10]:
## 9. Quick Sanity Checks

from src.db import get_recent_incidents, get_urgency_distribution, get_score_statistics

print("Total documents :", count_incidents())
print("\nUrgency distribution:")
for level, cnt in get_urgency_distribution().items():
    print(f"  {level:12s}: {cnt:,}")

print("\nThreat score stats:")
stats = get_score_statistics("threat_score")
for k, v in stats.items():
    print(f"  {k}: {v}")

print("\nMost recent 3 incidents:")
for doc in get_recent_incidents(n=3):
    print(f"  [{doc.get('cluster_id')}] {doc.get('title', '')[:80]}")

Total documents : 19204

Urgency distribution:
  medium      : 19,127
  low         : 68
  info        : 4
  high        : 3
  critical    : 2

Threat score stats:
  count: 19204
  avg: 45.4303
  min: 10.0
  max: 96.9
  std: 18.6304

Most recent 3 incidents:
  [971184ec] MikroTik RouterOS Vulnerabilities Under Active Exploitation
  [ec7534a8] Cyber Security Roles Address NHS Threats Amid Growing Risks
  [95d7c722] JetBrains Cadence Breach: Exploitation of Unpatched TeamCity CVE-2026-63077


In [11]:
## 10. Cleanup

close_connection()
print("MongoDB connection closed.")
print("Ingestion notebook finished successfully.")

MongoDB connection closed.
Ingestion notebook finished successfully.
